In [54]:

from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
import pandas as pd#
import pickle
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = 'Times New Roman'
mpl.rcParams['font.size'] = 10

In [15]:
trained_lora = {2222: None, 3333: None, 4444: None, 5555: None}

In [2]:
import time
from collections import deque
from typing import Dict, Tuple

import gymnasium as gym
import numpy as np
import torch
from torch import Tensor

from sample_factory.algo.learning.learner import Learner
from sample_factory.algo.sampling.batched_sampling import preprocess_actions
from sample_factory.algo.utils.action_distributions import argmax_actions
from sample_factory.algo.utils.env_info import extract_env_info
from sample_factory.algo.utils.make_env import make_env_func_batched
from sample_factory.algo.utils.misc import ExperimentStatus
from sample_factory.algo.utils.rl_utils import make_dones, prepare_and_normalize_obs
from sample_factory.algo.utils.tensor_utils import unsqueeze_tensor
from sample_factory.cfg.arguments import load_from_checkpoint
# from sample_factory.huggingface.huggingface_utils import generate_model_card, generate_replay_video, push_to_hf
from sample_factory.model.actor_critic import create_actor_critic
from sample_factory.model.model_utils import get_rnn_size
from sample_factory.utils.attr_dict import AttrDict
from sample_factory.utils.typing import Config, StatusCode
from sample_factory.utils.utils import debug_log_every_n, experiment_dir, log

# ---------------------------------------------------------------------------
# logging helpers (put near the top of the file, after imports)
# ---------------------------------------------------------------------------
import datetime, pathlib, json, pandas as pd, torch

def _ensure_parent(path: pathlib.Path):
    path.parent.mkdir(parents=True, exist_ok=True)

/home/fr/fr_lr554/.conda/envs/env/lib/python3.10/site-packages/deepmind_lab/__init__.py:26: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
%cd /work/classic/fr_lr554-TrainSpace/spectral_radius

/work/classic/fr_lr554-TrainSpace/spectral_radius


In [42]:
# from sample_factory.arguments import load_from_checkpoint
cfg_filename='/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNFixedSequenceLORA46_/04_RNNFixedSequenceLORA46_see_5555/config.json'
with open(cfg_filename, "r") as json_file:
    json_params = json.load(json_file)
    log.warning("Loading existing experiment configuration from %s", cfg_filename)
    loaded_cfg = AttrDict(json_params)

# # override the parameters in config file with values passed from command line
# for key, value in cfg.cli_args.items():
#     if key in loaded_cfg and loaded_cfg[key] != value:
#         log.debug("Overriding arg %r with value %r passed from command line", key, value)
#         loaded_cfg[key] = value

# # incorporate extra CLI parameters that were not present in JSON file
# for key, value in vars(cfg).items():
#     if key not in loaded_cfg:
#         log.debug("Adding new argument %r=%r that is not in the saved config file!", key, value)
#         loaded_cfg[key] = value

[2026-01-28 15:24:28,527][1245748] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNFixedSequenceLORA46_/04_RNNFixedSequenceLORA46_see_5555/config.json


In [43]:
cfg=loaded_cfg

In [44]:
import sys
from multiprocessing.context import BaseContext
from typing import Optional

from tensorboardX import SummaryWriter

from sample_factory.algo.runners.runner import AlgoObserver, Runner
from sample_factory.algo.utils.context import global_model_factory
from sample_factory.algo.utils.misc import ExperimentStatus
from sample_factory.algo.utils.multiprocessing_utils import get_mp_ctx
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sample_factory.train import make_runner
from sample_factory.utils.typing import Config, Env, PolicyID
from sample_factory.utils.utils import experiment_dir

# from sf_workingdir_lilly.dmlab.dmlab_env import (
#     DMLAB_ENVS,
#     dmlab_extra_episodic_stats_processing,
#     dmlab_extra_summaries,
#     list_all_levels_for_experiment,
#     make_dmlab_env,
# )
from sf_workingdir_lilly.dmlab.dmlab_level_cache import DmlabLevelCaches, make_dmlab_caches
# from sf_examples.dmlab.dmlab_model import make_dmlab_encoder
from sf_workingdir_lilly.dmlab.custom_core import make_hipposlam_core
from sf_workingdir_lilly.dmlab.custom_encoder import make_hipposlam_encoder
from sf_workingdir_lilly.dmlab.dmlab_params import add_dmlab_env_args, dmlab_override_defaults
from sf_workingdir_lilly.dmlab.custom_params import add_hipposlam_env_args, hipposlam_override_defaults


class DmlabEnvWithCache:
    def __init__(self, level_caches: Optional[DmlabLevelCaches] = None):
        self.caches = level_caches

    def make_env(self, env_name, cfg, env_config, render_mode) -> Env:
        return make_dmlab_env(env_name, cfg, env_config, render_mode, self.caches)


def register_dmlab_envs(level_caches: Optional[DmlabLevelCaches] = None):
    env_factory = DmlabEnvWithCache(level_caches)
    for env in DMLAB_ENVS:
        register_env(env.name, env_factory.make_env)


def register_dmlab_components(level_caches: Optional[DmlabLevelCaches] = None):
    # register_dmlab_envs(level_caches)
    global_model_factory().register_encoder_factory(make_hipposlam_encoder)
    global_model_factory().register_model_core_factory(make_hipposlam_core)


class DmlabExtraSummariesObserver(AlgoObserver):
    def extra_summaries(self, runner: Runner, policy_id: PolicyID, writer: SummaryWriter, env_steps: int) -> None:
        dmlab_extra_summaries(runner, policy_id, writer, env_steps)


def register_msg_handlers(cfg: Config, runner: Runner):
    if cfg.env == "dmlab_30":
        # extra functions to calculate human-normalized score etc.
        runner.register_episodic_stats_handler(dmlab_extra_episodic_stats_processing)
        runner.register_observer(DmlabExtraSummariesObserver())


def initialize_level_cache(cfg: Config, mp_ctx: BaseContext) -> Optional[DmlabLevelCaches]:
    if not cfg.dmlab_use_level_cache:
        return None

    env_name = cfg.env
    num_policies = cfg.num_policies if hasattr(cfg, "num_policies") else 1
    all_levels = list_all_levels_for_experiment(env_name)
    level_cache_dir = cfg.dmlab_level_cache_path
    caches = make_dmlab_caches(experiment_dir(cfg), all_levels, num_policies, level_cache_dir, mp_ctx)
    return caches


def parse_dmlab_args(argv=None, evaluation=False):
    parser, cfg = parse_sf_args(argv, evaluation=evaluation)
    add_hipposlam_env_args(parser)
    add_dmlab_env_args(parser)
    hipposlam_override_defaults(parser)
    cfg = parse_full_cfg(parser, argv)
    return cfg


# def main():
#     """Script entry point."""
#     cfg = parse_dmlab_args()

#     # explicitly create the runner instead of simply calling run_rl()
#     # this allows us to register additional message handlers
#     cfg, runner = make_runner(cfg)
#     register_msg_handlers(cfg, runner)

#     level_caches = initialize_level_cache(cfg, get_mp_ctx(cfg.serial_mode))
#     register_dmlab_components(level_caches)
        

#     status = runner.init()
#     if status == ExperimentStatus.SUCCESS:
#         status = runner.run()

#     return status


# if __name__ == "__main__":
#     sys.exit(main())


In [45]:
import sys

# from sample_factory.enjoy import enjoy
# from sf_workingdir_lilly.dmlab.train_hipposlam import parse_dmlab_args, register_dmlab_components

mapname="openfield_map2_fixed_loc3"
expname='hipposlam/RNNFixedSequenceLORA42_/00_RNNFixedSequenceLORA42_see_1111'

cli = [
    "--algo", "APPO",
    "--env", mapname ,         # pick any DM‑Lab level you have
    "--experiment", expname,
    "--encoder_load_path","/home/fr/fr_lr554/best_000025288_203030528_reward_94.185.pth",
    "--train_dir", "./train_dir", # anything writable
    "--max_num_frames", "50000",          # short rollout for the test
    "--num_envs", "8",
    "--dmlab_level_cache_path","./.dmlab_cache",
    "--load_checkpoint_kind","latest",
    "--use_jit","False",
    "--with_pos_obs","True",
    "--no_render",        # <-- skip human window; avoid X11 on servers
]

cli_dict={
 'algo': 'APPO',
 'env': mapname,
 'experiment': expname,
 'encoder_load_path': '/home/fr/fr_lr554/best_000025288_203030528_reward_94.185.pth',
 'train_dir': './train_dir',
 'max_num_frames': '50000',
 'num_envs': '8',
 'dmlab_level_cache_path': './.dmlab_cache',
 'load_checkpoint_kind': 'latest',
 'no_render': True,
 'use_jit': False,
 'with_pos_obs': True,
}
register_dmlab_components()
cfg = parse_dmlab_args(evaluation=True, argv=cli)

# tweak whatever you like *after* parsing
# cfg.with_pos_obs = True
cfg.cli_args=cli_dict
# status = enjoy(cfg)

[2026-01-28 15:24:30,497][1245748] register_encoder_factory: <function make_hipposlam_encoder at 0x7f3993c67be0>
[2026-01-28 15:24:30,497][1245748] register_model_core_factory: <function make_hipposlam_core at 0x7f39a0402f80>


In [46]:

REDUCED_ACTION_SET = (
    (0, 0, 0, 1, 0, 0, 0),  # Forward
    # (0, 0, 0, -1, 0, 0, 0),  # Backward
    (0, 0, -1, 0, 0, 0, 0),  # Strafe Left
    (0, 0, 1, 0, 0, 0, 0),  # Strafe Right
    # (-20, 0, 0, 0, 0, 0, 0),  # Look Left
    # (20, 0, 0, 0, 0, 0, 0),  # Look Right
    (-20, 0, 0, 1, 0, 0, 0),  # Look Left + Forward
    (20, 0, 0, 1, 0, 0, 0),  # Look Right + Forward
    # (0, 0, 0, 0, 1, 0, 0),  # Fire.
)
action_space = gym.spaces.Discrete(len(REDUCED_ACTION_SET))
observation_space = gym.spaces.Dict(
    obs=gym.spaces.Box(low=0, high=255, shape=(72, 96, 3), dtype=np.uint8)
)

In [47]:
    verbose = False

    cfg = load_from_checkpoint(cfg)

    eval_env_frameskip: int = cfg.env_frameskip if cfg.eval_env_frameskip is None else cfg.eval_env_frameskip
    assert (
        cfg.env_frameskip % eval_env_frameskip == 0
    ), f"{cfg.env_frameskip=} must be divisible by {eval_env_frameskip=}"
    render_action_repeat: int = cfg.env_frameskip // eval_env_frameskip
    cfg.env_frameskip = cfg.eval_env_frameskip = eval_env_frameskip
    log.debug(f"Using frameskip {cfg.env_frameskip} and {render_action_repeat=} for evaluation")

    cfg.num_envs = 1

    render_mode = "human"
    if cfg.save_video:
        render_mode = "rgb_array"
    elif cfg.no_render:
        render_mode = None

    # env = make_env_func_batched(
    #     cfg, env_config=AttrDict(worker_index=0, vector_index=0, env_id=0), render_mode=render_mode
    # )
    # env_info = extract_env_info(env, cfg)

    # if hasattr(env.unwrapped, "reset_on_init"):
    #     # reset call ruins the demo recording for VizDoom
    #     env.unwrapped.reset_on_init = False
    # log.info(env.action_space)
    actor_critic = create_actor_critic(cfg, observation_space, action_space)
    actor_critic.eval()

[2026-01-28 15:24:31,561][1245748] Loading existing experiment configuration from ./train_dir/hipposlam/RNNFixedSequenceLORA42_/00_RNNFixedSequenceLORA42_see_1111/config.json
[2026-01-28 15:24:31,561][1245748] Overriding arg 'experiment' with value 'hipposlam/RNNFixedSequenceLORA42_/00_RNNFixedSequenceLORA42_see_1111' passed from command line
[2026-01-28 15:24:31,562][1245748] Overriding arg 'train_dir' with value './train_dir' passed from command line
[2026-01-28 15:24:31,562][1245748] Overriding arg 'dmlab_level_cache_path' with value './.dmlab_cache' passed from command line
[2026-01-28 15:24:31,563][1245748] Overriding arg 'use_jit' with value False passed from command line
[2026-01-28 15:24:31,563][1245748] Overriding arg 'with_pos_obs' with value True passed from command line
[2026-01-28 15:24:31,564][1245748] Adding new argument 'fps'=0 that is not in the saved config file!
[2026-01-28 15:24:31,564][1245748] Adding new argument 'eval_env_frameskip'=None that is not in the saved 

ActorCriticSharedWeights(
  (obs_normalizer): ObservationNormalizer(
    (running_mean_std): RunningMeanStdDictInPlace(
      (running_mean_std): ModuleDict(
        (obs): RunningMeanStdInPlace()
      )
    )
  )
  (returns_normalizer): RecursiveScriptModule(original_name=RunningMeanStdInPlace)
  (encoder): HipposlamEncoder(
    (depth_encoder): DepthEncoder(
      (downsample): Upsample(size=(1, 10), mode='nearest')
    )
    (basic_encoder): ResnetEncoder(
      (conv_head): Sequential(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        (2): ResBlock(
          (res_block_core): Sequential(
            (0): ReLU()
            (1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (2): ReLU()
            (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          )
        )
        (3): ResBlock(
          (res_b

In [48]:
actor_critic.named_parameters( )

<generator object Module.named_parameters at 0x7f39934e51c0>

In [49]:
dict(actor_critic.named_modules()).keys()

dict_keys(['', 'obs_normalizer', 'obs_normalizer.running_mean_std', 'obs_normalizer.running_mean_std.running_mean_std', 'obs_normalizer.running_mean_std.running_mean_std.obs', 'returns_normalizer', 'encoder', 'encoder.depth_encoder', 'encoder.depth_encoder.downsample', 'encoder.basic_encoder', 'encoder.basic_encoder.conv_head', 'encoder.basic_encoder.conv_head.0', 'encoder.basic_encoder.conv_head.1', 'encoder.basic_encoder.conv_head.2', 'encoder.basic_encoder.conv_head.2.res_block_core', 'encoder.basic_encoder.conv_head.2.res_block_core.0', 'encoder.basic_encoder.conv_head.2.res_block_core.1', 'encoder.basic_encoder.conv_head.2.res_block_core.2', 'encoder.basic_encoder.conv_head.2.res_block_core.3', 'encoder.basic_encoder.conv_head.3', 'encoder.basic_encoder.conv_head.3.res_block_core', 'encoder.basic_encoder.conv_head.3.res_block_core.0', 'encoder.basic_encoder.conv_head.3.res_block_core.1', 'encoder.basic_encoder.conv_head.3.res_block_core.2', 'encoder.basic_encoder.conv_head.3.res_b

In [50]:
sd = actor_critic.state_dict()
# inspect keys
keys = [k for k in sd.keys() if k.startswith('core')]
print(keys[:50])

['core.rnn.weight_ih_l0', 'core.rnn.weight_hh_l0', 'core.rnn.lr_column', 'core.rnn.lr_row']


In [51]:
print(actor_critic.core.rnn.weight_hh_l0)
print(actor_critic.core.rnn.lr_column)
print(actor_critic.core.rnn.lr_row)

Parameter containing:
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.]])
Parameter containing:
tensor([[ 0.0292, -0.0028,  0.0187,  ...,  0.0184,  0.0018,  0.0196],
        [ 0.0051,  0.0093, -0.0136,  ...,  0.0126,  0.0228,  0.0210],
        [ 0.0212,  0.0228, -0.0154,  ..., -0.0115,  0.0083,  0.0176],
        ...,
        [ 0.0260, -0.0091,  0.0091,  ..., -0.0215, -0.0044, -0.0266],
        [ 0.0138, -0.0178,  0.0188,  ...,  0.0282, -0.0276,  0.0099],
        [ 0.0018, -0.0177,  0.0232,  ...,  0.0018,  0.0267, -0.0164]],
       requires_grad=True)
Parameter containing:
tensor([[-0.0282,  0.0210, -0.0178,  ..., -0.0095, -0.0243, -0.0213],
        [ 0.0081,  0.0221, -0.0287,  ..., -0.0175, -0.0285, -0.0167],
        [ 0.0064,  0.0286,  0.0140,  ...,  0.0007,  0.0054, -0.0084],
        .

In [52]:
trained_lora[5555]= {'lr_column': actor_critic.core.rnn.lr_column.detach().cpu().numpy(),
                    'lr_row': actor_critic.core.rnn.lr_row.detach().cpu().numpy()}

In [53]:
print(trained_lora[5555])

{'lr_column': array([[ 0.02921334, -0.00280157,  0.01873802, ...,  0.0183882 ,
         0.00183749,  0.01963752],
       [ 0.00514885,  0.00931208, -0.01358921, ...,  0.01261472,
         0.02280075,  0.02099602],
       [ 0.02122301,  0.02276896, -0.01536751, ..., -0.0115021 ,
         0.00828716,  0.01761277],
       ...,
       [ 0.02600616, -0.00910752,  0.00906852, ..., -0.02148215,
        -0.00437746, -0.02661917],
       [ 0.01383654, -0.01780676,  0.0187986 , ...,  0.02816673,
        -0.02761902,  0.00989853],
       [ 0.00184554, -0.01774335,  0.0232275 , ...,  0.00183843,
         0.02673442, -0.01640104]], dtype=float32), 'lr_row': array([[-0.02822668,  0.02100549, -0.01782828, ..., -0.00948009,
        -0.02427957, -0.02132629],
       [ 0.00805265,  0.02210259, -0.0286674 , ..., -0.01752352,
        -0.02854971, -0.01667984],
       [ 0.00641875,  0.0285743 ,  0.01399992, ...,  0.00067226,
         0.00543559, -0.00838929],
       ...,
       [ 0.00328235,  0.00620647,  

In [55]:
# save the extracted data
with open("/home/fr/fr_lr554/samplefactory/sample-factory/sf_workingdir_lilly/dmlab/analysis/data/sim46_lora.pkl", "wb") as f:
    pickle.dump(trained_lora, f)

In [ ]:
########################## STOP ##########################

In [ ]:
import torch.nn as nn

In [ ]:
print(type(actor_critic.core))
print(isinstance(actor_critic.core, nn.Module))

print("children:", list(actor_critic.core.named_children()))
print("modules :", list(actor_critic.core.named_modules())[:10])
print("params  :", list(actor_critic.core.named_parameters())[:10])

<class 'sf_examples.dmlab.Hipposlam_model.SimpleSequenceWithBypassCore'>
True
children: []
modules : [('', SimpleSequenceWithBypassCore())]
params  : []


In [ ]:
actor_critic.core.named_parameters()

dict_keys([])

In [ ]:
dict_keys(['', 'obs_normalizer', 'obs_normalizer.running_mean_std', 'obs_normalizer.running_mean_std.running_mean_std', 'obs_normalizer.running_mean_std.running_mean_std.obs', 'returns_normalizer', 'encoder', 'encoder.depth_encoder', 'encoder.depth_encoder.downsample', 'encoder.basic_encoder', 'encoder.basic_encoder.conv_head', 'encoder.basic_encoder.conv_head.0', 'encoder.basic_encoder.conv_head.1', 'encoder.basic_encoder.conv_head.2', 'encoder.basic_encoder.conv_head.2.res_block_core', 'encoder.basic_encoder.conv_head.2.res_block_core.0', 'encoder.basic_encoder.conv_head.2.res_block_core.1', 'encoder.basic_encoder.conv_head.2.res_block_core.2', 'encoder.basic_encoder.conv_head.2.res_block_core.3', 'encoder.basic_encoder.conv_head.3', 'encoder.basic_encoder.conv_head.3.res_block_core', 'encoder.basic_encoder.conv_head.3.res_block_core.0', 'encoder.basic_encoder.conv_head.3.res_block_core.1', 'encoder.basic_encoder.conv_head.3.res_block_core.2', 'encoder.basic_encoder.conv_head.3.res_block_core.3', 'encoder.basic_encoder.conv_head.4', 'encoder.basic_encoder.conv_head.5', 'encoder.basic_encoder.conv_head.6', 'encoder.basic_encoder.conv_head.6.res_block_core', 'encoder.basic_encoder.conv_head.6.res_block_core.0', 'encoder.basic_encoder.conv_head.6.res_block_core.1', 'encoder.basic_encoder.conv_head.6.res_block_core.2', 'encoder.basic_encoder.conv_head.6.res_block_core.3', 'encoder.basic_encoder.conv_head.7', 'encoder.basic_encoder.conv_head.7.res_block_core', 'encoder.basic_encoder.conv_head.7.res_block_core.0', 'encoder.basic_encoder.conv_head.7.res_block_core.1', 'encoder.basic_encoder.conv_head.7.res_block_core.2', 'encoder.basic_encoder.conv_head.7.res_block_core.3', 'encoder.basic_encoder.conv_head.8', 'encoder.basic_encoder.conv_head.9', 'encoder.basic_encoder.conv_head.10', 'encoder.basic_encoder.conv_head.10.res_block_core', 'encoder.basic_encoder.conv_head.10.res_block_core.0', 'encoder.basic_encoder.conv_head.10.res_block_core.1', 'encoder.basic_encoder.conv_head.10.res_block_core.2', 'encoder.basic_encoder.conv_head.10.res_block_core.3', 'encoder.basic_encoder.conv_head.11', 'encoder.basic_encoder.conv_head.11.res_block_core', 'encoder.basic_encoder.conv_head.11.res_block_core.0', 'encoder.basic_encoder.conv_head.11.res_block_core.1', 'encoder.basic_encoder.conv_head.11.res_block_core.2', 'encoder.basic_encoder.conv_head.11.res_block_core.3', 'encoder.basic_encoder.conv_head.12', 'encoder.basic_encoder.mlp_layers', 'encoder.basic_encoder.mlp_layers.0', 'encoder.DG_projection', 'encoder.DG_projection.linear', 'encoder.DG_projection.batchnorm1d', 'encoder.DG_projection.activation', 'core', 'decoder', 'decoder.mlp', 'decoder.mlp.0', 'decoder.mlp.1', 'decoder.mlp.2', 'critic_linear', 'action_parameterization', 'action_parameterization.distribution_linear'])

In [ ]:
[x for x in dict(actor_critic.named_modules()).keys() if x.startswith('encoder.DG')]